In [1]:
from march.module import DMC
import torch

In [2]:
# import os
# import subprocess
# import shutil
# import importlib
# import sys

# # Clean build directory
# build_path = "/home/koi/Documents/git/march/build/lib.linux-x86_64-cpython-310"
# if os.path.exists(build_path):
#     shutil.rmtree(build_path)
#     print(f"Removed {build_path}")

# # Rebuild extension
# os.chdir("/home/koi/Documents/git/march")
# result = subprocess.run(["python", "setup.py", "build_ext", "--inplace"], capture_output=True, text=True)
# print("Build succeeded!" if result.returncode == 0 else f"Build failed with code {result.returncode}")

# # Remove the module from cache if it's already loaded
# if 'march' in sys.modules:
#     del sys.modules['march']
# if 'march._C' in sys.modules:
#     del sys.modules['march._C']


In [3]:
"""
//  Coordinate system
//
//       z
//       |
//       |
//       |
//       0-----x
//      /
//     /
//    y
//
"""

'\n//  Coordinate system\n//\n//       z\n//       |\n//       |\n//       |\n//       0-----x\n//      /\n//     /\n//    y\n//\n'

In [4]:
"""
Edge and vertex convention:

                    v4_____________________v5_____________________v10
                    /|                    /|                     /|
                   / |                   / |                    / |
                  /  |                  /  |                   /  |
                 /___|_________________/___|__________________/   |
              v6|    |                 |v7 |                  |v11|
                |    |                 |   |                  |   |
                |    |                 |   |                  |   |
                |    |                 |   |                  |   |
                |    |_________________|___|__________________|___|
                |   /|v0               |   / v1               |   / v8
                |  / |                 |  /|                  |  /|
                | /  |                 | / |                  | / | 
                |/___|_________________|/__|__________________|/v9|    
              v2|    |                 |v3 |                  |   |
                |    |                 |   |                  |   |
                |    |                 |   |                  |   |
                |    |                 |   |                  |   |
                |    |_________________|___|__________________|___|
                |   / v12              |   / v13              |   / v16
                |  /                   |  /                   |  / 
                | /                    | /                    | /  
                |/_____________________|/_____________________|/
                v14                    v15                     v17
"""

'\nEdge and vertex convention:\n\n                    v4_____________________v5_____________________v10\n                    /|                    /|                     /|\n                   / |                   / |                    / |\n                  /  |                  /  |                   /  |\n                 /___|_________________/___|__________________/   |\n              v6|    |                 |v7 |                  |v11|\n                |    |                 |   |                  |   |\n                |    |                 |   |                  |   |\n                |    |                 |   |                  |   |\n                |    |_________________|___|__________________|___|\n                |   /|v0               |   / v1               |   / v8\n                |  / |                 |  /|                  |  /|\n                | /  |                 | / |                  | / | \n                |/___|_________________|/__|__________________|

Binary to Offset Table

| Index    | Binray Offset $(z, y, x)$ | Offset $(dx, dy, dz)$ |
| -------- | -------| -------- |
| 0        | 000    | $(0, 0, 0)$
| 1        | 001    | $(1, 0, 0)$
| 2        | 010    | $(0, 1, 0)$
| 3        | 011    | $(1, 1, 0)$
| 4        | 100    | $(0, 0, 1)$
| 5        | 101    | $(1, 0, 1)$
| 6        | 110    | $(0, 1, 1)$
| 7        | 111    | $(1, 1, 1)$

In [5]:
grids = torch.tensor([
    [0, 0, 0], #v0
    [1, 0, 0], #v1
    [0, 1, 0], #v2
    [1, 1, 0], #v3

    [0, 0, 1], #v4
    [1, 0, 1], #v5
    [0, 1, 1], #v6
    [1, 1, 1], #v7

    [2, 0, 0], #v8
    [2, 1, 0], #v9
    [2, 0, 1], #v10
    [2, 1, 1], #v11
    
    [0, 0, -1], #v12
    [1, 0, -1], #v13
    [0, 1, -1], #v14
    [1, 1, -1], #v15

    [2, 0, -1], #v16
    [2, 1, -1]  #v17
], dtype=torch.float32)

cubes = torch.tensor([
    [0, 1, 2, 3, 4, 5, 6, 7],
    [1, 8, 3, 9, 5, 10, 7, 11],
    [12, 13, 14, 15, 0, 1, 2, 3],
    [13, 16, 15, 17, 1, 8, 3, 9]
], dtype=torch.int32)

values = torch.tensor(
    [
        1,        #v0
        -0.5,     #v1
        1,        #v2
        -0.5,     #v3
        1,        #v4
        1,        #v5
        1,        #v6
        1,        #v7
        1,        #v8
        1,        #v9
        1,        #v10
        1,        #v11
        1,        #v12
        -0.5,     #v13
        1,        #v14
        -0.5,     #v15
        1,        #v16
        1         #v17
    ], dtype=torch.float32
)

grids = grids.contiguous().cuda()
cubes = cubes.contiguous().cuda()
values = values.contiguous().cuda()

iso = 0.0

mc = DMC()

verts, faces = mc(grids, cubes, values, iso)

print("# verts:", verts.shape[0])
print("# faces:", faces.shape[0])

print("Verts:")
print(verts)

print("Faces:")
print(faces)

# verts: 10
# faces: 8
Verts:
tensor([[ 0.6667,  0.0000,  0.0000],
        [ 1.0000,  0.0000,  0.3333],
        [ 1.3333,  0.0000,  0.0000],
        [ 0.6667,  1.0000,  0.0000],
        [ 1.0000,  1.0000,  0.3333],
        [ 1.3333,  1.0000,  0.0000],
        [ 0.6667,  0.0000, -1.0000],
        [ 1.3333,  0.0000, -1.0000],
        [ 0.6667,  1.0000, -1.0000],
        [ 1.3333,  1.0000, -1.0000]], device='cuda:0')
Faces:
tensor([[0, 3, 4],
        [1, 0, 4],
        [5, 2, 1],
        [4, 5, 1],
        [6, 8, 0],
        [8, 3, 0],
        [7, 2, 9],
        [9, 2, 5]], device='cuda:0')


In [6]:
import plotly.graph_objects as go

# Create figure
fig = go.Figure()

# Create color array: red if value > iso, blue otherwise
colors = ['red' if v > iso else 'blue' for v in values]

grids_numpy = grids.detach().cpu().numpy()
verts_numpy = verts.detach().cpu().numpy()
faces_numpy = faces.detach().cpu().numpy()

# Add grid points as scatter plot
fig.add_trace(go.Scatter3d(
    x=grids_numpy[:, 0],
    y=grids_numpy[:, 1],
    z=grids_numpy[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    name='Grid Points'
))

# Add mesh as wireframe
# First, create the mesh surface with low opacity
fig.add_trace(go.Mesh3d(
    x=verts_numpy[:, 0],
    y=verts_numpy[:, 1],
    z=verts_numpy[:, 2],
    i=faces_numpy[:, 0],
    j=faces_numpy[:, 1],
    k=faces_numpy[:, 2],
    opacity=0.1,
    color='lightblue',
    showlegend=False
))

# Add wireframe edges
edges_x = []
edges_y = []
edges_z = []
for i, j, k in faces_numpy:
    # Edge 1: vertex i to j
    edges_x.extend([verts_numpy[i, 0], verts_numpy[j, 0], None])
    edges_y.extend([verts_numpy[i, 1], verts_numpy[j, 1], None])
    edges_z.extend([verts_numpy[i, 2], verts_numpy[j, 2], None])
    # Edge 2: vertex j to k
    edges_x.extend([verts_numpy[j, 0], verts_numpy[k, 0], None])
    edges_y.extend([verts_numpy[j, 1], verts_numpy[k, 1], None])
    edges_z.extend([verts_numpy[j, 2], verts_numpy[k, 2], None])
    # Edge 3: vertex k to i
    edges_x.extend([verts_numpy[k, 0], verts_numpy[i, 0], None])
    edges_y.extend([verts_numpy[k, 1], verts_numpy[i, 1], None])
    edges_z.extend([verts_numpy[k, 2], verts_numpy[i, 2], None])

fig.add_trace(go.Scatter3d(
    x=edges_x,
    y=edges_y,
    z=edges_z,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name='Mesh Edges',
    showlegend=True
))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    title='Grid Points and Marching Cubes Mesh (Wireframe)',
    width=800,
    height=800
)

fig.show()


In [7]:
# import torch
# from march._C import MCF

# # Test direct binding call
# print("Testing direct binding call...")
# mc_direct = MCF()
# verts_direct, faces_direct = mc_direct.forward(grids, cubes, values, iso)
# print(f"Direct binding result: {verts_direct.shape[0]} vertices, {faces_direct.shape[0]} faces")
# print(f"Direct vertices:\n{verts_direct}")
# print(f"Direct faces:\n{faces_direct}")
# print()

# # Test module wrapper
# print("Testing module wrapper...")
# verts_module, faces_module = mc(grids, cubes, values, iso)
# print(f"Module wrapper result: {verts_module.shape[0]} vertices, {faces_module.shape[0]} faces")
# print(f"Module vertices:\n{verts_module}")
# print(f"Module faces:\n{faces_module}")
# print()

# # Compare
# print(f"Vertices match: {torch.allclose(verts_direct, verts_module, atol=1e-5)}")
# print(f"Faces match: {torch.allclose(faces_direct.float(), faces_module.float())}")

In [8]:
# # Verify backward pass works
# grids_bg = grids.clone().requires_grad_(True)
# values_bg = values.clone().requires_grad_(True)

# mc_bg = DMC()
# verts_bg, faces_bg = mc_bg(grids_bg, cubes, values_bg, iso)

# # Compute loss and backprop
# loss = verts_bg.sum() + faces_bg.float().sum()
# loss.backward()

# print("Backward pass test:")
# print(f"  Gradient computed for grid_vertices: {grids_bg.grad is not None}")
# print(f"  Gradient computed for values: {values_bg.grad is not None}")
# if values_bg.grad is not None:
#     print(f"  Values gradient shape: {values_bg.grad.shape}")
#     print(f"  Values gradient min/max: {values_bg.grad.min():.4f} / {values_bg.grad.max():.4f}")
# print("✓ Backward pass successful!")